# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [ ]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [ ]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [ ]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [ ]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [ ]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, then alternate between arching your back up (cat) and letting it sag down (cow). Aim for 10-15 repetitions.\n\n- **Bird Dog:** From your hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for about 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds, then repeat 8-12 times.\n\nThese exercises, performed gently and regularly, can help alleviate lower back discomfort and strengthen the muscles supporting your spine. Be sure to consult with a healthcare professional before starting any new exercise routine, especially if you have ongoing back issues.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical repair, mental well-being, and cognitive function. It helps the body repair tissues, regulate hormones related to growth and appetite, and consolidate memories. Adequate sleep, typically 7-9 hours per night for adults, also strengthens the immune system, improves mood, and enhances learning and memory. Poor sleep or conditions like insomnia can negatively impact these functions, leading to health issues. Maintaining good sleep hygiene and creating a conducive sleep environment are important strategies to promote better sleep and, consequently, better overall health.'

In [ ]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing deep breathing, progressive muscle relaxation, or grounding techniques for immediate stress relief\n- Taking short walks, especially in nature\n- Listening to calming music\n\nThese methods can help manage stress and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [ ]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future issues.'

In [ ]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate and quality sleep—typically 7 to 9 hours per night—supports various bodily functions. During sleep, the body goes through cycles, including REM and non-REM stages, with deep sleep (Stage 3) being crucial for body repair and regeneration. Proper sleep helps regulate immune function, mental health, memory, and learning. Additionally, creating an optimal sleep environment—such as maintaining a comfortable temperature, darkness, and quietness—can enhance sleep quality, further contributing to overall wellness. Conversely, issues like insomnia can negatively affect health by disrupting these vital processes.'

In [ ]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as deep breathing and progressive muscle relaxation, engaging in meditation, and consuming herbal teas like chamomile or valerian root. Managing hydration, ensuring adequate sleep, and reducing triggers like eye strain and certain foods can also help prevent headaches related to stress.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

*Your answer here*

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [ ]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

In [ ]:
import ssl
import truststore
import httpx
import cohere
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

ctx = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
httpx_client = httpx.Client(verify=ctx, timeout=60.0)

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"], httpx_client=httpx_client)
compressor = CohereRerank(model="rerank-v3.5", client=co)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [ ]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include: \n\n- Cat-Cow Stretch: Start on your hands and knees, then alternate between arching your back up (cat) and lowering it down (cow). Aim for 10-15 repetitions.\n- Bird Dog: From a hands and knees position, extend opposite arm and leg, keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle exercises can help alleviate and prevent lower back discomfort.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly affects overall health by supporting physical repair, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7-9 hours for adults—also involves various stages, including deep sleep where body regeneration occurs. Poor sleep or conditions like insomnia can impair these processes, highlighting the importance of creating a comfortable sleep environment to maintain overall health.'

In [ ]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying a cold or warm compress to the head or neck, resting in a dark and quiet room, giving gentle massages to the temples and neck, using essential oils like peppermint or lavender, maintaining a regular sleep schedule, practicing deep breathing exercises, progressive muscle relaxation, grounding techniques, taking short walks in nature, and listening to calming music.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [ ]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on your hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Repeat 8-12 times.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n\nThese exercises are gentle and aimed at relieving lower back discomfort and preventing future issues.'

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate and quality sleep (typically 7-9 hours for adults) supports a strong immune system, helps manage stress, and promotes recovery from illness. Poor sleep or sleep disorders like insomnia can lead to health problems such as increased stress, headaches, fatigue, digestive issues, and a weakened immune system. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are important practices to support overall health and well-being.'

In [ ]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils such as peppermint or lavender\n- Engaging in relaxation techniques like deep breathing (e.g., inhale for 4 counts, hold, exhale), progressive muscle relaxation, or grounding exercises\n- Taking a short walk, preferably in nature\n- Listening to calming music\n- Practicing mindfulness and meditation regularly\n- Maintaining a regular sleep schedule and practicing good sleep hygiene\n\nThese approaches can help alleviate the symptoms naturally and promote overall well-being.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

*Your answer here*

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [ ]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [ ]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [ ]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [ ]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"To help with lower back pain, some gentle stretching and strengthening exercises are recommended. These include:\n\n- **Cat-Cow Stretch:** On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs and tilt your pelvis up slightly to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help allevi

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep—typically 7-9 hours per night for adults—is essential for maintaining energy levels, immune function, and mental clarity. Poor sleep or sleep deprivation can lead to fatigue, decreased immune response, impaired memory, and increased risk of chronic health conditions. Therefore, prioritizing good sleep hygiene and creating an optimal sleep environment are important practices for overall health.'

In [ ]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, doing mindfulness or meditation, taking short walks in nature, listening to calming music, and using essential oils like peppermint or lavender. Additionally, maintaining a consistent sleep schedule, managing stress through hobbies and social support, and staying well-hydrated can help alleviate headaches related to stress.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [ ]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [ ]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis slightly upwards. Hold for 10 seconds and repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\nRemember to perform these exer

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it supports physical repair, mental well-being, and cognitive functioning. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night, helps strengthen the immune system, improve mood, and maintain intellectual performance. Poor sleep or insomnia can negatively impact physical health, increase stress levels, and hinder mental clarity. Maintaining good sleep hygiene, such as creating a restful environment and sticking to a consistent sleep schedule, is crucial for reaping these health benefits.'

In [ ]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing exercises (such as inhaling for 4 counts, holding, exhaling, and holding again), progressive muscle relaxation (tensing and relaxing muscle groups), grounding techniques (naming things you see, hear, feel, smell, and taste), taking short walks in nature, and listening to calming music. \n\nFor headaches, natural remedies include staying well-hydrated by drinking water, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule. \n\nThese approaches can help manage stress and headaches naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [ ]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [ ]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [ ]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [ ]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future issues.'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep is essential for overall health because it allows the body to repair tissues, consolidate memories, and regulate hormone release related to growth and appetite. Adequate sleep (typically 7-9 hours for adults) supports physical and mental well-being, enhances cognitive function, and helps maintain a healthy immune system. Poor sleep or sleep disturbances can negatively impact energy levels, increase stress, impair memory, and elevate the risk for various health issues. Therefore, maintaining good sleep hygiene and ensuring restful sleep are vital for promoting overall health.'

In [ ]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Staying well-hydrated by drinking plenty of water\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of temples and neck muscles\n- Using essential oils such as peppermint or lavender\n- Practicing relaxation techniques like deep breathing exercises and progressive muscle relaxation\n- Engaging in short walks outdoors, preferably in nature\n- Listening to calming music\n- Incorporating mindfulness and meditation practices to reduce overall stress\n\nThese approaches can help manage stress and alleviate headaches naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

*Your answer here*

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE

# enable langsmith tracing

import os
import getpass
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
os.environ["LANGCHAIN_PROJECT"] = "ai_bootcamp"

In [ ]:
# use sgd to create the dataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_3596\981573009.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_3596\981573009.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
df = dataset.to_pandas()
df = df.drop(columns=["persona_name", "query_style", "query_length"], errors="ignore")
df

,user_input,reference_contexts,reference,synthesizer_name
0,How do I do exersize and movemnt to help with ...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Exercise is important for health and can help ...,single_hop_specific_query_synthesizer
1,What is Chaptr 1?,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Chapter 1: Understanding Exercise Basics\n\nEx...,single_hop_specific_query_synthesizer
2,How does sleep contribute to overall health an...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,"Sleep is crucial for physical health, mental w...",single_hop_specific_query_synthesizer
3,What are the potential benefits of using valer...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Herbal teas such as chamomile or valerian root...,single_hop_specific_query_synthesizer
4,Can you tell me about Chapter 17 and what it s...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"Chapter 17 discusses digestive health, emphasi...",single_hop_specific_query_synthesizer
5,What does PART 5 refer to in building healthy ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 5 refers to the section on building healt...,single_hop_specific_query_synthesizer
6,How does Chapter 12 on mindfulness and meditat...,[<1-hop>\n\nPART 4: STRESS MANAGEMENT AND MENT...,Chapter 12 focuses on mindfulness and meditati...,multi_hop_specific_query_synthesizer
7,How does Chapter 13's discussion on building h...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 13 explains that building healthy habi...,multi_hop_specific_query_synthesizer
8,Chapter 11 talk about stress reduction techniq...,[<1-hop>\n\nPART 4: STRESS MANAGEMENT AND MENT...,Chapter 11 discusses stress reduction techniqu...,multi_hop_specific_query_synthesizer
9,Considering the comprehensive information prov...,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,"Maintaining a balanced diet, as outlined in Ch...",multi_hop_specific_query_synthesizer


In [68]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
import copy, time

custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

retrievers = {
    "naive": naive_retriever,
    "bm25": bm25_retriever,
    "multi_query": multi_query_retriever,
    "parent_doc": parent_document_retriever,
    "compression": compression_retriever,
    "ensemble": ensemble_retriever,
}

metrics = [
    LLMContextRecall(),
    ContextPrecision(),
    ContextEntityRecall(),
    NoiseSensitivity()
]

results = {}

for name, retriever in retrievers.items():
    dataset_copy = df.copy()

    retrieved_contexts = []
    for _, row in dataset_copy.iterrows():
        q = row["user_input"]  # <- DataFrame column, not row.eval_sample.user_input

        # use whichever your retriever supports
        docs = retriever.invoke(q) if hasattr(retriever, "invoke") else retriever.get_relevant_documents(q)

        retrieved_contexts.append([d.page_content for d in docs])
        time.sleep(0.2)

    dataset_copy["retrieved_contexts"] = retrieved_contexts

    # Some Ragas versions expect response to exist even for retriever metrics
    if "response" not in dataset_copy.columns:
        dataset_copy["response"] = ""

    eval_ds = EvaluationDataset.from_pandas(dataset_copy)
    results[name] = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    time.sleep(60)  # to avoid hitting rate limits

results

C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_3596\99767055.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_3596\99767055.py:2: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_3596\99767055.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metri

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[3]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[7]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[11]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[15]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[19]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[23]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[27]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[31]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[35]: Value

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[3]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[7]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[11]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[15]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[19]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[23]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[27]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[31]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[35]: Value

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[3]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[7]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[11]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[15]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[19]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[23]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[27]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[31]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[35]: Value

Evaluating:   0%|          | 0/48 [00:00<?, ?it/s]

Exception raised in Job[3]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[7]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[11]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[15]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[19]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[23]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[27]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[31]: ValueError(response is missing in the test sample. Please add response to the test sample.)
Exception raised in Job[35]: Value

TooManyRequestsError: status_code: 429, body: data=None id='7eac7d8f-8e17-4c73-aa1c-3571017b7eb3' message="You are using a Trial key, which is limited to 10 API calls / minute. You can continue to use the Trial key for free or upgrade to a Production key with higher rate limits at 'https://dashboard.cohere.com/api-keys'. Contact us on 'https://discord.gg/XW44jPfYJu' or email us at support@cohere.com with any questions"

In [ ]:
from langsmith import traceable

@traceable(name="retrieve_only")
def run_retriever(retriever, question: str, k: int = 4):
    t0 = time.perf_counter()
    docs = retriever.invoke(question)  # or get_relevant_documents(question)
    latency_s = time.perf_counter() - t0
    return {
        "question": question,
        "latency_s": latency_s,
        "num_docs": len(docs),
        "contexts": [d.page_content for d in docs],
    }
    
questions = [row["question"] for _, row in dataset.iterrows()]  # your golden set

for q in questions:
    _ = run_retriever(naive_retriever, q)
    _ = run_retriever(bm25_retriever, q)
    _ = run_retriever(ensemble_retriever, q)
    _ = run_retriever(parent_document_retriever, q)
    _ = run_retriever(compression_retriever, q)
    _ = run_retriever(multi_query_retriever, q)

In [ ]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

In [ ]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )